# Milestone 3 — evaluate the trained v5 adapter and paired-test it

The v5 adapter already exists: `nyaya-train-v5` trained it in 2.39h and saved
`checkpoint-75`, then died in its evaluation cell on a missing gitignored input.
This notebook consumes that kernel's output as an INPUT so the 2.4h of
training is not repeated.

**Settings:** GPU **T4 x2**, Internet **On**, and add `jitendrajha98/nyaya-train-v5`
as a notebook data source (Add Input -> Notebook Output).

Runs ~1h: generate v5 predictions on Eval-v1, then bootstrap the paired
difference against the committed base predictions.

In [ ]:
# --- setup + preflight -------------------------------------------------
!pip -q install -U transformers accelerate peft rank_bm25 sentence-transformers "torchao>=0.16.0"

import os

# One GPU: Kaggle's T4 x2 makes HF pick DataParallel and split tensors across
# cuda:0/cuda:1 while the model is pinned to one device.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import glob, subprocess, sys, time

import torch

def run(cmd, check=True):
    print("$", " ".join(str(c) for c in cmd), flush=True)
    p = subprocess.run([str(c) for c in cmd], text=True)
    if check and p.returncode != 0:
        raise RuntimeError(f"step failed: {' '.join(str(c) for c in cmd)}")
    return p.returncode

REPO = "https://github.com/JitendraJha98/nyaya-model.git"
if not os.path.exists("/kaggle/working/nyaya-model"):
    subprocess.run(["git", "clone", "--depth", "1", REPO,
                    "/kaggle/working/nyaya-model"], check=True)
os.chdir("/kaggle/working/nyaya-model")
sys.path.insert(0, "src")

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings -> Accelerator -> GPU T4 x2.")
major, minor = torch.cuda.get_device_capability(0)
arch = f"sm_{major}{minor}"
print(f"GPU: {torch.cuda.get_device_name(0)} ({arch}) | visible: {torch.cuda.device_count()}")
if arch not in torch.cuda.get_arch_list():
    raise RuntimeError(f"{arch} unsupported by this torch build")
print("preflight OK")

In [ ]:
# --- locate the trained adapter from /kaggle/input ---------------------
# Layout-agnostic on purpose: the adapter may arrive as a kernel output
# (nested under outputs/legal-3b-v5/checkpoint-N) or as a dataset (flat), and
# Kaggle refuses kernel_sources from a kernel that errored — which the
# training kernel did, after training succeeded. Find it by its weights file.
weights = glob.glob("/kaggle/input/**/adapter_model.safetensors", recursive=True)
if not weights:
    raise RuntimeError(
        "v5 adapter not found under /kaggle/input. Add Input -> Dataset -> "
        "jitendrajha98/nyaya-3b-v5-adapter")
# Prefer the highest checkpoint number when several are present.
def _ckpt_rank(path):
    import re
    m = re.search(r"checkpoint-(\d+)", path)
    return int(m.group(1)) if m else 0

ADAPTER = os.path.dirname(max(weights, key=_ckpt_rank))
print("adapter:", ADAPTER)
print("contents:", sorted(os.listdir(ADAPTER))[:8])

import json as _json
cfg_path = os.path.join(ADAPTER, "adapter_config.json")
assert os.path.exists(cfg_path), "adapter_config.json missing next to weights"
_cfg = _json.load(open(cfg_path))
print(f"base_model: {_cfg.get('base_model_name_or_path')} | r={_cfg.get('r')}")

# Rebuild the gitignored inputs. Eval-v1 first — its absence is exactly what
# killed the training kernel after 2.4h of successful work.
run([sys.executable, "scripts/25_build_eval_v1.py"])
import pathlib
for required in ("data/eval/nyaya_eval_v1.jsonl",
                 "outputs/eval-v1/base/predictions.jsonl"):
    assert pathlib.Path(required).exists(), f"missing input: {required}"
print("all inputs present")

In [ ]:
# --- SMOKE: 8 questions through the adapter, timed --------------------
# Time GENERATION, not the model load. Wrapping the whole subprocess measured
# 24.8 s/question on a run whose own generation clock said 8.8 -- the ~60s
# load dominated 8 questions and tripped this guard. scripts/26 prints
# "wall clock: Ns" for the batch loop only, after the model is resident, so
# parse that instead of re-deriving it badly.
import re

proc = subprocess.run(
    [sys.executable, "scripts/26_eval_v1_run.py", "--adapter", ADAPTER,
     "--dense", "--k", "8", "--limit", "8", "--batch-size", "4",
     "--label", "v5-smoke"],
    text=True, capture_output=True)
print(proc.stdout[-1500:])
if proc.returncode != 0:
    print(proc.stderr[-2000:])
    raise RuntimeError("smoke eval failed")

m = re.search(r"wall clock\s*:\s*([\d.]+)s", proc.stdout)
if not m:
    raise RuntimeError("could not parse generation wall clock from smoke output")
gen_s = float(m.group(1))
per_q = gen_s / 8
projected = per_q * 413 / 60
print(f"\ngeneration only: {gen_s:.0f}s for 8 questions -> {per_q:.1f}s/question")
print(f"projected for 413: ~{projected:.0f} min")
if projected > 120:
    raise RuntimeError(f"~{projected:.0f} min exceeds budget — lower --k or "
                       f"raise --batch-size")
print("smoke OK")

In [ ]:
# --- full v5 evaluation on Eval-v1 ------------------------------------
# Same retriever, same k, same questions as the committed base run — the only
# variable is the adapter. That is what makes the paired test meaningful.
t0 = time.time()
run([sys.executable, "scripts/26_eval_v1_run.py", "--adapter", ADAPTER,
     "--dense", "--k", 8, "--split", "all", "--batch-size", 4,
     "--label", "nyaya-3b-v5"])
print(f"eval wall clock: {(time.time() - t0)/60:.0f} min")

In [ ]:
# --- THE CLAIM TEST: paired bootstrap vs base -------------------------
# If the 95% CI excludes zero, "better than base" is defensible. If it spans
# zero, it is not — and that gets reported exactly as it comes out.
run([sys.executable, "scripts/27_compare_runs.py", "--a", "base", "--b", "nyaya-3b-v5"])

In [ ]:
# --- collect everything, including the adapter ------------------------
import glob
import pathlib
import shutil

out = pathlib.Path("/kaggle/working/nyaya-v5-results")
out.mkdir(exist_ok=True)
for f in ("reports/eval_v1_results.json", *glob.glob("reports/eval_v1_comparison_*.json"),
          "reports/v5_dataset_report.json"):
    if os.path.exists(f):
        shutil.copy(f, out)
for pred in pathlib.Path("outputs/eval-v1").glob("*/predictions.jsonl"):
    shutil.copy(pred, out / f"{pred.parent.name}_predictions.jsonl")
# The adapter is the deliverable — carry it out of the input mount so it lives
# in this run's output too, not only the training kernel's.
shutil.copytree(ADAPTER, out / "adapter", dirs_exist_ok=True)
shutil.make_archive("/kaggle/working/nyaya-v5-results", "zip", out)
print("collected:", sorted(p.name for p in out.iterdir()))
print("Download nyaya-v5-results.zip from the Output tab.")